# **CSB314: Exploratory Data Analysis — Minor Project 02 (CA 3)**
## **Data Analysis of Brazilian E-Commerce Public Dataset (Olist)**

**Name:** Rimshi Alam  
**Course:** CSB314 (Exploratory Data Analysis)  
**Topics Covered:** Unit IV (Feature Engineering, Grouping, Reshaping & Data Integration) and Unit V (Descriptive Statistics, Distributions, Sampling/CLT, Correlation & Outlier Detection)


## **1. Problem Statement and Project Objectives**

### **Main Question**
How do product prices, shipping charges, delivery time, and product categories affect customer orders and review scores in the Brazilian e-commerce dataset (Olist) from 2016 to 2018?

### **Objectives of This Notebook**
1. **Check Data Quality:** Find missing values, duplicate rows, wrong data types, and key mismatches across tables.
2. **Create New Features:** Make useful columns from existing data like total item cost, freight ratio, delivery days, weekend flag, and review rating groups.
3. **Group & Summarize Data:** Use `groupby()`, `agg()`, `transform()`, `pivot_table()`, and `crosstab()` to compare product categories and states.
4. **Reshape Tables:** Convert data between wide and long formats using `melt()`, `pivot()`, `stack()`, and `unstack()`.
5. **Join Multiple Tables Properly:** Combine related tables using `merge()`, `join()`, and `concat()` without multiplying rows by mistake.
6. **Descriptive Statistics & Distributions:** Calculate mean, median, mode, variance, standard deviation, quartiles, skewness, and kurtosis, and check the 68-95-99.7 Empirical Rule and Central Limit Theorem (CLT).
7. **Correlation & Outliers:** Check relationships using covariance, Pearson, and Spearman correlation, and detect outliers using IQR and Z-score methods.


## **2. Dataset Source and Files Used**

* **Dataset Name:** Brazilian E-Commerce Public Dataset by Olist
* **Kaggle Link:** https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce
* **About the Data:** Contains around 100,000 real anonymized orders from Brazil between 2016 and 2018.

### Files Included in `./data/` Folder
| File Name | What Each Row Means | Primary Key | Foreign Keys |
| :--- | :--- | :--- | :--- |
| `olist_orders_dataset.csv` | One customer order | `order_id` | `customer_id` |
| `olist_customers_dataset.csv` | One customer record per order | `customer_id` | `customer_unique_id` |
| `olist_order_items_dataset.csv` | One item inside an order | `(order_id, order_item_id)` | `order_id`, `product_id`, `seller_id` |
| `olist_order_payments_dataset.csv` | One payment entry for an order | `(order_id, payment_sequential)` | `order_id` |
| `olist_order_reviews_dataset.csv` | One customer review for an order | `review_id` | `order_id` |
| `olist_products_dataset.csv` | One product in the catalog | `product_id` | `product_category_name` |
| `olist_sellers_dataset.csv` | One seller on Olist | `seller_id` | — |
| `product_category_name_translation.csv` | Portuguese to English category name | `product_category_name` | — |


## **3. Importing Libraries and Loading Data**

### **Objective**
Import Python libraries (`pandas`, `numpy`, `matplotlib`, `seaborn`, `scipy.stats`) and load the raw CSV files into separate variables.

### **Methodology**
* We set `RANDOM_SEED = 42` so that whenever we take random samples, we get the exact same result every time.
* We keep raw DataFrames (`raw_orders`, `raw_items`, etc.) untouched and create copies (`df_orders`, `df_items`, etc.) for cleaning and analysis.
* `scipy.stats` is used to calculate Z-scores easily for outlier detection.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

DATA_DIR = './data/'

# Load untouched raw files
raw_orders = pd.read_csv(os.path.join(DATA_DIR, 'olist_orders_dataset.csv'))
raw_customers = pd.read_csv(os.path.join(DATA_DIR, 'olist_customers_dataset.csv'))
raw_items = pd.read_csv(os.path.join(DATA_DIR, 'olist_order_items_dataset.csv'))
raw_payments = pd.read_csv(os.path.join(DATA_DIR, 'olist_order_payments_dataset.csv'))
raw_reviews = pd.read_csv(os.path.join(DATA_DIR, 'olist_order_reviews_dataset.csv'))
raw_products = pd.read_csv(os.path.join(DATA_DIR, 'olist_products_dataset.csv'))
raw_sellers = pd.read_csv(os.path.join(DATA_DIR, 'olist_sellers_dataset.csv'))
raw_translation = pd.read_csv(os.path.join(DATA_DIR, 'product_category_name_translation.csv'))

print("All 8 CSV files loaded successfully!")


## Explaination

### 1. Importing Core Data Science Libraries (`os`, `numpy`, `pandas`, `matplotlib.pyplot`, `scipy.stats`)
* **What it does:** Imports essential Python packages required for file system navigation, numerical computations, tabular data manipulation, plotting, and statistical analysis.
* **Key Functions & Logic:**
  * **`os`**: Manages file directory paths and operating system interactions.
  * **`numpy` (`np`)**: Provides support for high-performance numerical arrays and mathematical operations.
  * **`pandas` (`pd`)**: The primary library for data structures, cleaning, merging, and tabular analysis.
  * **`matplotlib.pyplot` (`plt`)**: Enables the generation of charts, figures, and data visualizations.
  * **`scipy.stats`**: Supplies statistical functions and probability distributions for advanced data testing.

### 2. Defining Directory Paths and Loading Raw CSV Files (`pd.read_csv`, `os.path.join`)
* **What it does:** Sets the root data directory and reads all eight Olist Brazilian E-Commerce CSV files into separate Pandas dataframes.
* **Key Functions & Logic:**
  * **`os.path.join()`**: Constructively builds safe file paths across different operating systems by combining `DATA_DIR` with individual CSV filenames.
  * **`pd.read_csv()`**: Parses comma-separated text files into structured Pandas dataframes (`raw_orders`, `raw_customers`, `raw_items`, `raw_payments`, `raw_reviews`, `raw_products`, `raw_sellers`, `raw_translation`) to maintain an untouched baseline for the project pipeline.

### 3. Verification and Execution Output (`print`)
* **What it does:** Confirms that all datasets have been successfully imported into the active Python workspace without errors.
* **Key Functions & Logic:**
  * **`print()`**: Outputs a confirmation message to the console once all eight CSV files are loaded.

---

## **4. Basic Dataset Understanding**

### **Objective**
Check the number of rows, columns, and sample data in each CSV file.

### **Observation & Decision**
From `.head()`, we can see that all date and time columns in `raw_orders` are stored as text (`object`). We will convert them into proper `datetime` format in our working DataFrames.


In [ ]:
tables = {
    'orders': raw_orders,
    'customers': raw_customers,
    'order_items': raw_items,
    'order_payments': raw_payments,
    'order_reviews': raw_reviews,
    'products': raw_products,
    'sellers': raw_sellers,
    'category_translation': raw_translation
}

info_list = []
for name, df in tables.items():
    info_list.append({
        'Table': name,
        'Rows': df.shape[0],
        'Columns': df.shape[1]
    })

display(pd.DataFrame(info_list))
display(raw_orders.head(3))
display(raw_items.head(3))


## Explaination

### 1. Dictionary-Based Table Mapping (`dict`, `for` loop, `.shape`)
* **What it does:** Organizes all raw Olist datasets into a dictionary structure and iterates through them to dynamically compute the row and column dimensions of each table.
* **Key Functions & Logic:**
  * **Dictionary Definition (`tables = {...}`)**: Groups the raw dataframes under descriptive keys (`orders`, `customers`, `order_items`, etc.) for streamlined iteration.
  * **`.shape[0]` & `.shape[1]`**: Extracts the exact row count and column count for every table in the collection.

### 2. Compiling and Displaying the Shape Summary (`pd.DataFrame`, `display`)
* **What it does:** Converts the collected dimension metrics into a structured summary DataFrame and renders it alongside sample rows from the foundational tables.
* **Key Functions & Logic:**
  * **`pd.DataFrame(info_list)`**: Builds a clean, tabular overview comparing the scale and size of all eight relational tables.
  * **`.head(3)`**: Displays the first 3 rows of both `raw_orders` and `raw_items` to inspect data structure and column schemas.

---

## **5. Data Quality Assessment (Completeness, Consistency, Accuracy & Keys)**

### **Objective**
Check the data for missing values, duplicate rows, invalid numbers, wrong dates, and broken foreign keys before doing any calculations.

### **Concept**
* **Completeness:** How many values are missing (`NaN`) in each column.
* **Consistency:** Whether text categories (like state names or order status) use uniform spelling and uppercase/lowercase format.
* **Accuracy:** Checking domain rules such as `price > 0`, `freight_value >= 0`, and `review_score` between 1 and 5.
* **Key Integrity:** Checking if primary keys are unique and whether foreign keys match the parent table.


In [ ]:
# Check Missing Values (Completeness)
def get_missing(df, table_name):
    m_count = df.isnull().sum()
    m_pct = (m_count / len(df)) * 100
    res = pd.DataFrame({
        'Table': table_name,
        'Missing_Count': m_count,
        'Missing_Pct': m_pct.round(2),
        'Data_Type': df.dtypes
    })
    return res[res['Missing_Count'] > 0]

missing_summary = pd.concat([
    get_missing(raw_orders, 'orders'),
    get_missing(raw_items, 'order_items'),
    get_missing(raw_payments, 'order_payments'),
    get_missing(raw_reviews, 'order_reviews'),
    get_missing(raw_products, 'products')
])
print("--- Columns with Missing Values ---")
display(missing_summary)



## Explaination

### 1. Defining a Missing Value Audit Function (`isnull`, `sum`, `pd.DataFrame`)
* **What it does:** Creates a reusable function to compute the absolute count and percentage of missing values for any dataframe, merging the metrics with their respective data types into a structured audit table.
* **Key Functions & Logic:**
  * **`.isnull().sum()`**: Counts the number of null or missing entries per column.
  * **Missing Percentage Calculation (`(m_count / len(df)) * 100`)**: Computes the proportion of missing records relative to total rows, rounding to two decimal places.
  * **Filtering (`res['Missing_Count'] > 0`)**: Isolates and returns only columns that contain missing values, keeping the audit output concise.

### 2. Concatenating Missing Value Summaries Across Tables (`pd.concat`)
* **What it does:** Runs the audit function across multiple raw Olist tables (`orders`, `order_items`, `order_payments`, `order_reviews`, `products`) and stacks the results vertically.
* **Key Functions & Logic:**
  * **`pd.concat([...])`**: Combines the individual missing value summary DataFrames into a single, comprehensive audit report spanning all major dataset components.

### 3. Displaying the Audit Report (`print`, `display`)
* **What it does:** Prints a descriptive header and renders the final aggregated missing value summary table nicely within the notebook environment.
* **Key Functions & Logic:**
  * **`display()`**: Outputs the formatted `missing_summary` DataFrame for visual inspection of data quality and completeness gaps.

---

In [ ]:
# Check Duplicates, Primary Keys, Consistency & Invalid Values
print("--- Exact Duplicate Rows ---")
for name, df in tables.items():
    print(f"{name:22s}: {df.duplicated().sum()} duplicates")

print("\n--- Primary Key Uniqueness ---")
print("orders.order_id unique?      :", raw_orders['order_id'].is_unique)
print("customers.customer_id unique?:", raw_customers['customer_id'].is_unique)
print("products.product_id unique?  :", raw_products['product_id'].is_unique)
print("reviews.review_id unique?    :", raw_reviews['review_id'].is_unique, f"({raw_reviews['review_id'].nunique()} unique out of {len(raw_reviews)})")

print("\n--- Consistency & Plausibility Checks ---")
print("Order status values          :", raw_orders['order_status'].unique())
print("Payment types                :", raw_payments['payment_type'].unique())
print("Items with price <= 0        :", (raw_items['price'] <= 0).sum())
print("Items with freight < 0       :", (raw_items['freight_value'] < 0).sum())
print("Reviews outside 1 to 5       :", (~raw_reviews['review_score'].isin([1, 2, 3, 4, 5])).sum())

# 5.3 Check Foreign Keys
missing_order_fk = set(raw_items['order_id']) - set(raw_orders['order_id'])
missing_prod_fk = set(raw_items['product_id']) - set(raw_products['product_id'])
unmapped_cats = set(raw_products['product_category_name'].dropna()) - set(raw_translation['product_category_name'])

print("\n--- Foreign Key Checks ---")
print("Items with missing order_id  :", len(missing_order_fk))
print("Items with missing product_id:", len(missing_prod_fk))
print("Categories missing translation:", unmapped_cats)

### **Observations and Cleaning Decisions**

1. **Missing Dates:** About 2.98% of orders do not have `order_delivered_customer_date`. These are mostly `canceled` or `shipped` orders that were not delivered yet. **Decision:** Keep these rows as `NaN` so we do not lose valid order records.
2. **Missing Review Comments:** `review_comment_title` (~88% missing) and `review_comment_message` (~58% missing) are optional for customers. **Decision:** Keep all rows.
3. **Duplicate `review_id`:** A few `review_id` values appear more than once. We will group reviews by `order_id` before merging.
4. **Unmapped Product Categories:** Two categories (`pc_gamer` and `portateis_cozinha_e_preparadores_de_alimentos`) are not in the translation file. **Decision:** Add their English names manually so no products are lost during merging.


In [ ]:
# Make working copies (raw_ DataFrames remain untouched)
df_orders = raw_orders.copy()
df_customers = raw_customers.copy()
df_items = raw_items.copy()
df_payments = raw_payments.copy()
df_reviews = raw_reviews.copy()
df_products = raw_products.copy()
df_sellers = raw_sellers.copy()
df_translation = raw_translation.copy()

# Convert date columns to datetime
date_cols = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_cols:
    df_orders[col] = pd.to_datetime(df_orders[col], errors='coerce')

df_items['shipping_limit_date'] = pd.to_datetime(df_items['shipping_limit_date'], errors='coerce')

# Check if any order shows delivered before purchase date
bad_dates = df_orders[df_orders['order_delivered_customer_date'] < df_orders['order_purchase_timestamp']]
print("Orders delivered before purchase timestamp:", len(bad_dates))

# Clean text casing and add missing English category names
df_products['product_category_name'] = df_products['product_category_name'].fillna('unknown').str.strip().str.lower()
df_translation['product_category_name'] = df_translation['product_category_name'].str.strip().str.lower()
df_translation['product_category_name_english'] = df_translation['product_category_name_english'].str.strip().str.lower()

extra_cats = pd.DataFrame({
    'product_category_name': ['unknown', 'pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos'],
    'product_category_name_english': ['unknown', 'pc_gamer', 'portable_kitchen_food_preparers']
})
df_translation = pd.concat([df_translation, extra_cats], ignore_index=True).drop_duplicates('product_category_name')
print("Working DataFrames cleaned and ready.")


## Explaination

### 1. Creating Isolated Working Copies (`copy`)
* **What it does:** Duplicates all raw foundational tables into working dataframes (`df_orders`, `df_customers`, `df_items`, etc.) to safeguard the original raw data from accidental alterations or destructive transformations.
* **Key Functions & Logic:**
  * **`.copy()`**: Allocates separate memory for each working dataframe, ensuring operations performed during cleaning and feature engineering do not contaminate the baseline raw datasets.

### 2. Converting date columns to datetime (`pd.to_datetime`, `errors='coerce'`)
* **What it does:** Converts raw string-based date columns into structured Python datetime objects across both the orders and items tables.
* **Key Functions & Logic:**
  * **`pd.to_datetime()`**: Parses date and timestamp text strings into standardized datetime formats for accurate temporal calculations.
  * **`errors='coerce'`**: Gracefully handles parsing failures by converting unparseable dates or malformed text into missing values (`NaT`) instead of throwing a runtime error.

### 3. Data Integrity Validation and Anomaly Check (`datetime` filtering, `len`)
* **What it does:** Audits the order fulfillment timeline to identify logical contradictions, specifically checking for records where the delivery date precedes the purchase date.
* **Key Functions & Logic:**
  * **Comparison Mask (`df_orders['order_delivered_customer_date'] < df_orders['order_purchase_timestamp']`)**: Isolates chronologically impossible records where delivery happened before purchase.
  * **`len()`**: Counts and reports the number of anomalies found to assess data cleanliness.

### 4. String Cleaning and Translation Dictionary Expansion (`fillna`, `str.strip`, `str.lower`, `pd.concat`)
* **What it does:** Normalizes product category text fields by stripping whitespace and converting text to lowercase, fills missing categories, and expands the translation mapping table to handle unassigned codes.
* **Key Functions & Logic:**
  * **Text Normalization (`.str.strip().str.lower()`)**: Eliminates trailing/leading whitespaces and standardizes casing for reliable category joining.
  * **`fillna('unknown')`**: Replaces null category entries with a default 'unknown' placeholder string.
  * **Dictionary Augmentation (`pd.concat`)**: Appends custom English translations for unmapped category codes (like `pc_gamer` and `portateis_cozinha_e_preparadores_de_alimentos`) and drops duplicate entries to ensure clean category merging.

---

## **Task 1: Feature Engineering**

### **Objective & Concept**
Create useful analytical variables from existing raw data to make patterns easier to study. We derive mathematical, date/time, logical, and categorical features.

### **Details of New Features Created**
| New Feature | Columns Used | Formula / Rule | Data Type | Why We Created It |
| :--- | :--- | :--- | :--- | :--- |
| `item_total_value` | `price`, `freight_value` | `price + freight_value` | `float64` | Shows total cost paid for each item including shipping. |
| `freight_ratio` | `freight_value`, `price` | `freight_value / price` | `float64` | Shows how high shipping cost is compared to the item price. |
| `delivery_duration_days` | `order_delivered_customer_date`, `order_purchase_timestamp` | `(delivered - purchase) in days` | `float64` | Measures how many days an order took to reach the customer. |
| `is_weekend_purchase` | `order_purchase_timestamp` | `dayofweek >= 5` (Saturday or Sunday) | `bool` | Checks if customers order differently on weekends vs weekdays. |
| `review_score_category` | `review_score` | `5 -> High`, `3-4 -> Medium`, `1-2 -> Low` | `category` | Groups numeric ratings into easy-to-read satisfaction levels. |


In [ ]:
df_items = raw_items.copy()
df_orders = raw_orders.copy()
df_reviews = raw_reviews.copy()

# converting date columns  to datetime for .dt accessors
df_orders['order_purchase_timestamp'] = pd.to_datetime(df_orders['order_purchase_timestamp'])
df_orders['order_delivered_customer_date'] = pd.to_datetime(df_orders['order_delivered_customer_date'])

# 1. Mathematical features on items table
df_items['item_total_value'] = df_items['price'] + df_items['freight_value']
df_items['freight_ratio'] = np.where(df_items['price'] > 0, df_items['freight_value'] / df_items['price'], np.nan)

# 2. Date/Time & Logical features on orders table
df_orders['delivery_duration_days'] = (
    df_orders['order_delivered_customer_date'] - df_orders['order_purchase_timestamp']
).dt.total_seconds() / 86400.0

df_orders.loc[df_orders['delivery_duration_days'] < 0, 'delivery_duration_days'] = np.nan
df_orders['purchase_month'] = df_orders['order_purchase_timestamp'].dt.to_period('M').astype(str)
df_orders['is_weekend_purchase'] = df_orders['order_purchase_timestamp'].dt.dayofweek >= 5

# 3. Conditional categorical feature on reviews table
conds = [
    df_reviews['review_score'] == 5,
    df_reviews['review_score'].isin([3, 4]),
    df_reviews['review_score'].isin([1, 2])
]
labels = ['High (5)', 'Medium (3-4)', 'Low (1-2)']
df_reviews['review_score_category'] = pd.Categorical(
    np.select(conds, labels, default='Unknown'),
    categories=['Low (1-2)', 'Medium (3-4)', 'High (5)'],
    ordered=True
)

display(df_items[['order_id', 'price', 'freight_value', 'item_total_value']].head(3))
display(df_orders[['order_id', 'delivery_duration_days', 'is_weekend_purchase']].dropna().head(3))

## Task 1 explaination

### 1. Copying Dataframes and Date Conversion (`copy`, `pd.to_datetime`)
* **What it does:** Creates local copies of raw dataframes to prevent modifying the original data and converts order timestamp strings into true datetime objects to enable `.dt` accessor methods.
* **Key Functions & Logic:**
  * **`.copy()`**: Ensures data isolation by duplicating `raw_items`, `raw_orders`, and `raw_reviews`.
  * **`pd.to_datetime()`**: Parses date-time text strings into structured datetime formats for `order_purchase_timestamp` and `order_delivered_customer_date`.

### 2. Mathematical Features on Items Table (`item_total_value`, `freight_ratio`)
* **What it does:** Engineers numerical columns on the items dataframe to capture total financial cost per item and the relative proportion of shipping costs.
* **Key Functions & Logic:**
  * **`item_total_value`**: Sums up product `price` and `freight_value` to calculate total transaction cost per item.
  * **`np.where()`**: Implements conditional logic to compute `freight_ratio` (`freight_value / price`) only when `price > 0`, safely returning `np.nan` otherwise to prevent division-by-zero errors.

### 3. Date/Time & Logical Features on Orders Table (`delivery_duration_days`, `purchase_month`, `is_weekend_purchase`)
* **What it does:** Extracts time-based attributes from order dates, calculates fulfillment duration in days, handles data anomalies, and creates logical flags for weekend orders.
* **Key Functions & Logic:**
  * **Duration Calculation**: Subtracts purchase timestamp from delivery timestamp, converts total seconds to days (`/ 86400.0`), and masks illogical negative durations (`< 0`) as `np.nan`.
  * **`dt.to_period('M')`**: Truncates timestamps to monthly periods and converts them to strings (`purchase_month`) for monthly trend aggregation.
  * **`dt.dayofweek >= 5`**: Evaluates whether an order was placed on Saturday or Sunday (day 5 or 6), returning a boolean flag (`is_weekend_purchase`).

### 4. Conditional Categorical Feature on Reviews Table (`np.select`, `pd.Categorical`)
* **What it does:** Categorizes numerical customer review scores into ordinal text-based satisfaction levels ('Low', 'Medium', 'High') for qualitative analysis.
* **Key Functions & Logic:**
  * **`np.select()`**: Evaluates multiple boolean conditions (`conds`) mapped against corresponding text `labels` ('High (5)', 'Medium (3-4)', 'Low (1-2)'), assigning a default value of 'Unknown' if unmatched.
  * **`pd.Categorical()`**: Converts the resulting labels into a formal ordered categorical data type, explicitly defining the ranking category order (`categories=['Low (1-2)', 'Medium (3-4)', 'High (5)']`, `ordered=True`) for downstream ordinal operations.

### 5. Displaying Results (`display`, `dropna`, `head`)
* **What it does:** Outputs sample rows of the newly engineered item and order metrics to verify the transformation results.
* **Key Functions & Logic:**
  * **`dropna()`**: Filters out missing records before rendering order outputs.
  * **`head(3)`**: Limits the display view to the first 3 rows of each table.

---

## **Task 2: Grouping and Aggregation**

### **Objective & Concept**
**Aggregation (agg()):** 
Collapses the data by groups into a single summary row per group, which reduces the total row count. 

**Transformation (transform()):**
 Calculates a group-level statistic (like the average price of each category) and broadcasts it back to match every individual row in the original table, keeping the exact same row count so you can compare each row against its group average.

In [ ]:
# 1. Prepare enriched items dataframe by merging items, products, and category translations
df_items_enriched = raw_items.merge(raw_products, on='product_id', how='left')
df_items_enriched = df_items_enriched.merge(raw_translation, on='product_category_name', how='left')

# 2. Aggregation (reduces rows to summary groups)
cat_stats = (
    df_items_enriched
    .groupby('product_category_name_english')
    .agg(
        total_items=('order_item_id', 'count'),
        total_sales=('price', 'sum'),
        avg_price=('price', 'mean'),
        median_price=('price', 'median')
    )
    .sort_values(by='total_sales', ascending=False)
)

print("--- Top 10 Product Categories by Sales Summary ---")
display(cat_stats.head(10))

# 3. Transformation (broadcasts group statistics back to original row count)
df_items_enriched['category_avg_price'] = (
    df_items_enriched.groupby('product_category_name_english')['price'].transform('mean')
)

print("\n--- Sample of Enriched Items with Broadcasted Category Average Price ---")
display(df_items_enriched[['order_id', 'product_id', 'product_category_name_english', 'price', 'category_avg_price']].head(3))

# 4. Bar Chart Visualization for Top 10 Product Categories by Sales
top10_sales = cat_stats.head(10).sort_values('total_sales', ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(top10_sales.index, top10_sales['total_sales'] / 1e6, color='steelblue')
ax.set_title('Top 10 Product Categories by Total Sales Revenue (2016–2018)', fontsize=12, fontweight='bold')
ax.set_xlabel('Total Sales (Million BRL / R$)', fontsize=10)
ax.set_ylabel('Product Category', fontsize=10)

# Add value labels to the bars
for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.02, bar.get_y() + bar.get_height()/2, f'R$ {w:.2f}M', va='center', fontsize=9)

plt.tight_layout()
plt.show()

## Task 2 explaination

### 1. Preparing the Enriched Items Dataframe (`merge`)
* **What it does:** This step combines three separate datasets (`raw_items`, `raw_products`, and `raw_translation`) into a single unified table named `df_items_enriched`.
* **Key Functions & Logic:**
  * **`merge()`**: Joins dataframes based on common keys (similar to SQL JOINs).
  * **`how='left'`**: Performs a left outer join, ensuring that all rows from the primary table are retained, even if a matching product or translation record is missing.
  * **First Merge:** Links `raw_items` with `raw_products` using `product_id` to bring in product-specific attributes.
  * **Second Merge:** Links the resulting dataframe with `raw_translation` using `product_category_name` to map Portuguese category names into their English equivalents (`product_category_name_english`).

### 2. Grouping and Aggregation (`groupby` & `agg`)
* **What it does:** This step groups the item-level data by product category and condenses the rows to calculate key summary statistics for each category.
* **Key Functions & Logic:**
  * **`groupby('product_category_name_english')`**: Splits the dataset into distinct groups based on each English product category.
  * **`agg()` (Named Aggregation)**: Applies multiple summary functions simultaneously while explicitly naming the resulting columns (`total_items`, `total_sales`, `avg_price`, `median_price`).
  * **`sort_values(by='total_sales', ascending=False)`**: Ranks the categories from highest to lowest based on total sales revenue.

### 3. Data Transformation (`transform`)
* **What it does:** Unlike aggregation (which reduces the number of rows), **transformation** computes group-level statistics and **broadcasts** them back so they match the original dataframe's row count.
* **Key Functions & Logic:**
  * **`transform('mean')`**: Calculates the average price for each category just like `.agg()`, but maps the average back to every individual row belonging to that category instead of returning a summarized table.
  * **Use Case:** This allows row-level comparisons—such as comparing an individual product's price against its category's average price within the same dataframe.

### 4. Bar Chart Visualization (`Matplotlib`)
* **What it does:** Generates a clean, horizontal bar chart displaying the top 10 product categories ranked by total sales revenue.
* **Key Components & Logic:**
  * **`head(10).sort_values(..., ascending=True)`**: Selects the top 10 categories and sorts them in ascending order so the highest-selling category appears at the top of the horizontal chart.
  * **`ax.barh()`**: Creates a horizontal bar chart, giving ample space to read category labels clearly on the y-axis.
  * **Scaling (`/ 1e6`)**: Converts raw sales values into millions of Brazilian Real (BRL / R$) for better readability on the x-axis.
  * **Annotation Loop (`for bar in bars`)**: Iterates through each bar to dynamically extract its width and display the exact sales value label formatted as `R$ X.XXM`.

---

### **Group-Level Comparison :**

**Volume Leader (bed_bath_table):**  
Leads in total items sold (11,115 items) with an average price of R$ 93.30, showing it is a high-turnover, lower-cost category.

**Value Leader (health_beauty):** 
Achieves the highest total sales revenue (R$ 1,258,681.34) with a moderate average price of R$ 130.16.

**High-Ticket Category (watches_gifts):** 
Ranks second in total revenue (R$ 1,205,005.68) despite fewer unit sales (5,991 items) because its average price is much higher (R$ 201.14 per item).

## **Task 3: Pivot Tables and Crosstabulation**

### **Objective & Concept**

**Pivot Table (pivot_table()):** 
Summarizes a continuous numerical measure across two categorical dimensions using an aggregation function (like mean or sum), with margins to show overall totals.   

**Crosstabulation (crosstab()):** Crosses two categorical variables to examine their frequency or normalized proportions (percentages), helping us discover patterns between categories.

In [ ]:
if 'df_master' not in globals():
    order_items_agg = raw_items.groupby('order_id', as_index=False).agg(
        total_items=('order_item_id', 'count'),
        order_price_sum=('price', 'sum'),
        order_freight_sum=('freight_value', 'sum')
    )
    order_payments_agg = raw_payments.groupby('order_id', as_index=False).agg(
        payment_records=('payment_sequential', 'count'),
        total_payment_value=('payment_value', 'sum'),
        primary_payment_type=('payment_type', 'first')
    )
    df_master = pd.merge(raw_orders, raw_customers[['customer_id', 'customer_state']], on='customer_id', how='left')
    df_master = pd.merge(df_master, order_items_agg, on='order_id', how='left')
    df_master = pd.merge(df_master, order_payments_agg, on='order_id', how='left')
    df_master['is_weekend_purchase'] = pd.to_datetime(df_master['order_purchase_timestamp']).dt.dayofweek >= 5

# 1. Pivot Table: Average Freight Cost across Top Customer States and Payment Types
top_states = df_master['customer_state'].value_counts().head(6).index
sub_states = df_master[
    df_master['customer_state'].isin(top_states) &
    df_master['primary_payment_type'].isin(['credit_card', 'boleto', 'voucher', 'debit_card'])
]

freight_pivot = pd.pivot_table(
    sub_states,
    values='order_freight_sum',
    index='customer_state',
    columns='primary_payment_type',
    aggfunc='mean',
    margins=True,
    margins_name='Overall_Avg'
)

print("--- Pivot Table: Average Order Freight (BRL) by State & Payment Method ---")
display(freight_pivot.round(2))

# 2. Crosstabulation: Order Status vs Weekday/Weekend (Row-Normalized Proportions)
status_ct = pd.crosstab(
    index=df_master['order_status'],
    columns=df_master['is_weekend_purchase'].map({False: 'Weekday', True: 'Weekend'}),
    normalize='index',
    margins=True
) * 100

print("\n--- Crosstab: Order Status (%) on Weekdays vs Weekends ---")
display(status_ct.round(2))

## Task 3 explaination

### 1. Filtering Master Dataset and Creating a Pivot Table (`df_master`, `pivot_table`, `value_counts`)
* **What it does:** Leverages the integrated master dataset (`df_master`), filters for top customer states and common payment types, and generates a pivot table to analyze average freight costs across these dimensions.
* **Key Functions & Logic:**
  * **`df_master` Integration:** Relies on the pre-built master dataframe—which unifies orders, customer locations, pre-aggregated items, and payment records—to supply the necessary attributes (`customer_state`, `primary_payment_type`, and `order_freight_sum`).
  * **`value_counts().head(6).index`**: Identifies the top 6 most frequent customer states from the dataset.
  * **`.isin()`**: Filters rows to include only those matching the top states and specified payment methods (`credit_card`, `boleto`, `voucher`, `debit_card`).
  * **`pd.pivot_table()`**: Reshapes the filtered data, setting `customer_state` as rows, `primary_payment_type` as columns, and computing the average (`mean`) of `order_freight_sum`.
  * **`margins=True` & `margins_name='Overall_Avg'`**: Adds summary totals and row/column averages labeled as 'Overall_Avg' to the pivot table.

### 2. Crosstabulation of Order Status vs. Purchase Timing (`df_master`, `crosstab`, `normalize`)
* **What it does:** Pulls from `df_master` to compute row-normalized percentage distributions comparing how order fulfillment statuses vary between weekday and weekend purchases.
* **Key Functions & Logic:**
  * **Dataset Dependency (`df_master`)**: Accesses engineered temporal features (like `is_weekend_purchase`) and fulfillment metrics (`order_status`) directly from the unified master table.
  * **`.map({False: 'Weekday', True: 'Weekend'})`**: Converts boolean weekend flags into readable string labels.
  * **`pd.crosstab()`**: Computes a frequency table between `order_status` and the mapped weekday/weekend column.
  * **`normalize='index'`**: Converts absolute frequencies into row-wise proportions summing to 1.0.
  * **Multiplication (`* 100`)**: Converts the normalized proportions into percentage format for easier interpretation.

### 3. Formatting and Displaying Results (`round`, `display`)
* **What it does:** Cleans up the numerical output by rounding decimals and prints the summarized analytical tables to the console.
* **Key Functions & Logic:**
  * **`.round(2)`**: Rounds all floating-point numbers in both the pivot table and crosstab to two decimal places for presentation clarity.
  * **`display()`**: Renders the formatted `freight_pivot` and `status_ct` tables nicely within the notebook environment.

---

## **Task 4: Data Reshaping**

### **Objective & Concept**

**Wide Format vs. Long Format:**

Wide format stores different measured variables in separate columns (each metric has its own column).

Long format stores variable names and values down in rows (metric names in one column, values in another).

### **Why Reshaping Helps:**

**Analysis & Plotting:** Many plotting libraries (like Seaborn) and statistical models need tidy long-format data to plot multiple categories or metrics correctly.

**Integration:** Reshaping helps restructure summary tables so they can easily be merged or joined with other datasets.

In [ ]:
# Ensuring delivery_duration_days exists in df_master
if 'delivery_duration_days' not in df_master.columns:
    df_master['order_purchase_timestamp'] = pd.to_datetime(df_master['order_purchase_timestamp'])
    df_master['order_delivered_customer_date'] = pd.to_datetime(df_master['order_delivered_customer_date'])
    df_master['delivery_duration_days'] = (
        df_master['order_delivered_customer_date'] - df_master['order_purchase_timestamp']
    ).dt.total_seconds() / 86400.0

# 1. Create a wide summary table grouped by state
wide_df = (
    df_master[df_master['customer_state'].isin(['SP', 'RJ', 'MG', 'RS', 'PR'])]
    .groupby('customer_state', as_index=False)
    .agg(
        avg_price=('order_price_sum', 'mean'),
        avg_freight=('order_freight_sum', 'mean'),
        avg_delivery_days=('delivery_duration_days', 'mean')
    )
)
print("--- 1. Original Wide Format Table ---")
display(wide_df)

# 2. Demonstrate melt() to convert wide format into long format rows
long_df = pd.melt(
    wide_df,
    id_vars=['customer_state'],
    value_vars=['avg_price', 'avg_freight', 'avg_delivery_days'],
    var_name='metric_name',
    value_name='metric_value'
)
print("\n--- 2. Long Format Table using pd.melt() ---")
display(long_df.head(8))

# 3. Demonstrate pivot() to reshape long format back into a wide format table
pivoted_df = long_df.pivot(
    index='customer_state',
    columns='metric_name',
    values='metric_value'
)
print("\n--- 3. Wide Format Restored using .pivot() ---")
display(pivoted_df)

# 4. Demonstrate stack() and unstack() on a MultiIndex Series
multi_idx_series = (
    df_master[df_master['customer_state'].isin(['SP', 'RJ', 'MG']) & df_master['primary_payment_type'].isin(['credit_card', 'boleto'])]
    .groupby(['customer_state', 'primary_payment_type'])['total_payment_value']
    .mean()
)

# unstack() converts a MultiIndex Series into a wide DataFrame
unstacked_df = multi_idx_series.unstack(level='primary_payment_type')

# stack() converts the wide DataFrame back into a MultiIndex Series
restacked_series = unstacked_df.stack()

print("\n--- 4. .unstack() Output (Series to Wide DataFrame) ---")
display(unstacked_df)

print("\n--- 5. .stack() Output (Wide DataFrame back to MultiIndex Series) ---")
display(restacked_series)

## Task 4 explaination

### Prerequisite Check (`delivery_duration_days`)
* **What it does:** Verifies if the `delivery_duration_days` column exists in the master dataframe, and computes it from purchase and delivery timestamps if it is missing.
* **Key Functions & Logic:**
  * **`if 'delivery_duration_days' not in df_master.columns`**: Ensures robustness by checking for the required column before proceeding.
  * **`pd.to_datetime()`**: Parses timestamp columns and calculates delivery duration in days by finding the difference in seconds and dividing by `86400.0` (seconds per day).

### 1. Creating a Wide Summary Table (`groupby`, `agg`)
* **What it does:** Filters the dataset for key states ('SP', 'RJ', 'MG', 'RS', 'PR') and aggregates average price, freight, and delivery duration into a wide summary dataframe.
* **Key Functions & Logic:**
  * **`.isin()`**: Filters rows matching the specified top customer states.
  * **`groupby(..., as_index=False)`**: Groups data by state while keeping `customer_state` as a standard column rather than an index.
  * **Named Aggregation (`agg`)**: Computes the mean for `order_price_sum`, `order_freight_sum`, and `delivery_duration_days` under custom column names.

### 2. Reshaping Wide to Long Format (`pd.melt`)
* **What it does:** Converts the wide summary table into a long format where multiple metric columns are unpivoted into rows.
* **Key Functions & Logic:**
  * **`pd.melt()`**: Unpivots a dataframe from wide to long format.
  * **`id_vars=['customer_state']`**: Keeps `customer_state` as the fixed identifier column.
  * **`value_vars`**: Specifies which metric columns to melt down into rows (`avg_price`, `avg_freight`, `avg_delivery_days`).
  * **`var_name` & `value_name`**: Renames the resulting variable and value columns to `metric_name` and `metric_value`.

### 3. Reshaping Long Back to Wide Format (`pivot`)
* **What it does:** Restores the long-format dataframe back into a wide format layout using index, columns, and values.
* **Key Functions & Logic:**
  * **`long_df.pivot()`**: Reshapes the dataframe based on column values.
  * **`index='customer_state'`**: Sets states as the rows of the new table.
  * **`columns='metric_name'`**: Spreads the unique values of `metric_name` out into distinct columns.
  * **`values='metric_value'`**: Populates the table cells using the numerical metric values.

### 4. Hierarchical Index Manipulation (`unstack` and `stack`)
* **What it does:** Computes a MultiIndex Series grouped by state and payment type, then demonstrates pivoting structures using `.unstack()` and `.stack()`.
* **Key Functions & Logic:**
  * **MultiIndex Groupby**: Calculates mean `total_payment_value` grouped by both `customer_state` and `primary_payment_type`.
  * **`.unstack(level='primary_payment_type')`**: Pivots the inner level of a MultiIndex Series, transforming it into a wide-format DataFrame with payment types as columns.
  * **`.stack()`**: Performs the reverse operation, pivoting columns back down into a hierarchical MultiIndex Series.

---

## **Task 5: Data Integration and Grain Management**

### **Objective & Concept**

**Row Level / Structure:** Each table in a dataset represents a specific level of detail (for example, one row per order vs. one row per individual item).

**Preventing Row Multiplication:** One order (order_id) can have multiple items and multiple payments. If we join child tables directly to the parent orders table without aggregating first,rows multiply together ($M \times N$), inflating the financial totals and averages.
 
**Solution:** First, summarize child tables so each has 1 row per order ID before combining them, and use merge validation to check the work.

### **Detailed Explanation of Integration Concepts**

**Primary Keys (PK) & Foreign Keys (FK):**

* **Primary Key:** orders.order_id is the Primary Key of the orders table.
* **Foreign Key:** order_items.order_id acts as a Foreign Key pointing back to orders.order_id.   

**Relationship Type:** The relationship between orders and order_items is one-to-many (one order can contain multiple items).   

### **Join Types Explained:**

**Inner Join (how='inner'):** Retains only the rows where the matching key exists in both tables.  
**Left Join (how='left'):** Retains all rows from the left table, and matching rows from the right table (unmatched right keys become NaN).  
**Right Join (how='right'):** Retains all rows from the right table, and matching rows from the left table.  
**Outer Join (how='outer'):** Retains all rows from both tables, filling missing matches with NaN.

### **Suffixes and Index Alignment:**

**Suffixes:**
If two tables share non-key column names, we use parameters like suffixes=('_left', '_right') during .merge() to tell them apart.   
**Index Alignment (.join()):** Unlike .merge() which matches columns, DataFrame .join() aligns and merges two DataFrames using their index labels automatically.   

### **Unmatched Records & Row Count Validation:**

Using indicator=True creates an extra column (_merge_indicator) that tells us exactly which rows matched (both) and which only existed in one table, confirming zero unexpected row loss.   

In [ ]:
# 1. Check Primary Key Uniqueness Before Merging
print("--- Key Uniqueness Checks ---")
print("orders.order_id unique?      :", df_orders['order_id'].is_unique)
print("customers.customer_id unique?:", df_customers['customer_id'].is_unique)
print("products.product_id unique?  :", df_products['product_id'].is_unique)

# 2. Summarize child tables to 1 row per order_id before merging
# This prevents accidental row multiplication (Cartesian product inflation)
order_items_agg = df_items.groupby('order_id', as_index=False).agg(
    total_items=('order_item_id', 'count'),
    order_price_sum=('price', 'sum'),
    order_freight_sum=('freight_value', 'sum')
)

order_payments_agg = df_payments.groupby('order_id', as_index=False).agg(
    payment_records=('payment_sequential', 'count'),
    total_payment_value=('payment_value', 'sum'),
    primary_payment_type=('payment_type', 'first')
)

print(f"\nRow count of df_orders       : {len(df_orders):,}")
print(f"Row count of order_items_agg : {len(order_items_agg):,}")

# 3. Integrate tables using pd.merge() with validation and indicator
rows_before = len(df_orders)

df_master = pd.merge(
    df_orders,
    df_customers[['customer_id', 'customer_city', 'customer_state']],
    on='customer_id',
    how='left',
    validate='one_to_one'
)

# Merge summarized items using indicator=True to track matches
df_master = pd.merge(
    df_master,
    order_items_agg,
    on='order_id',
    how='left',
    validate='one_to_one',
    indicator='_merge_indicator'
)

rows_after = len(df_master)
print(f"\nRow count before merge       : {rows_before:,}")
print(f"Row count after merge        : {rows_after:,}")
print("\n--- Match Indicator Breakdown ---")
print(df_master['_merge_indicator'].value_counts())

# Merge payments
df_master = pd.merge(
    df_master,
    order_payments_agg,
    on='order_id',
    how='left',
    validate='one_to_one'
)

# 4. Demonstrate join() using Index Alignment
# .join() combines DataFrames based on their matching index rather than columns
orders_indexed = df_orders.set_index('order_id')[['order_status', 'order_purchase_timestamp']]
payments_indexed = order_payments_agg.set_index('order_id')[['total_payment_value', 'primary_payment_type']]

indexed_joined_df = orders_indexed.join(payments_indexed, how='inner')
print("--- Index-Aligned .join() Result ---")
print(f"Shape of joined DataFrame by index: {indexed_joined_df.shape}")
display(indexed_joined_df.head(3))

# 5. Demonstrate concat() for vertical stacking
# Combines two subsets vertically (e.g., combining 2017 and 2018 orders)
orders_2017 = df_master[df_master['order_purchase_timestamp'].dt.year == 2017].head(3)
orders_2018 = df_master[df_master['order_purchase_timestamp'].dt.year == 2018].head(3)

stacked_df = pd.concat([orders_2017, orders_2018], axis=0, keys=['Year_2017', 'Year_2018'])
print("--- pd.concat() Vertical Stack Example ---")
display(stacked_df[['order_id', 'order_status', 'order_price_sum']])

## Task 5 explaination

### 1. Primary Key Uniqueness Verification (`is_unique`)
* **What it does:** Checks whether key identifier columns across core dataframes contain strictly unique values prior to merging, preventing unexpected data duplication.
* **Key Functions & Logic:**
  * **`.is_unique`**: Evaluates uniqueness on primary key columns (`order_id`, `customer_id`, `product_id`), returning a boolean value (`True`/`False`).

### 2. Child Table Summarization and Aggregation (`groupby`, `agg`)
* **What it does:** Reduces multi-row child tables (`df_items`, `df_payments`) down to a single row per `order_id` to prevent data inflation or Cartesian product explosions during merging.
* **Key Functions & Logic:**
  * **`groupby('order_id', as_index=False)`**: Groups transactional rows by order identifier while keeping `order_id` as a column.
  * **Named Aggregation (`agg`)**: Summarizes items into total counts, prices, and freight sums, and payment records into total payment values and primary payment types using functions like `count`, `sum`, and `first`.

### 3. Integrated Merging with Validation and Indicators (`pd.merge`, `validate`, `indicator`)
* **What it does:** Securely integrates customer, item, and payment data into a master order table using left joins backed by relationship checks and match tracking.
* **Key Functions & Logic:**
  * **`validate='one_to_one'`**: Enforces data integrity by raising an error if the merge relationships violate a strict one-to-one mapping.
  * **`indicator='_merge_indicator'`**: Appends a tracking column to audit merge match results (e.g., confirming whether rows matched successfully).
  * **Row Count Monitoring**: Tracks table length before and after integration to verify structural consistency.

### 4. Index-Aligned Joining (`set_index`, `join`)
* **What it does:** Demonstrates DataFrame combination using row indices instead of explicit merge keys, simulating an index-based inner join.
* **Key Functions & Logic:**
  * **`.set_index('order_id')`**: Replaces default integer index positions with `order_id` for both order and payment subsets.
  * **`.join(..., how='inner')`**: Combines dataframes side-by-side using their aligned matching indexes, printing the resulting shape and top rows.

### 5. Vertical Data Stacking (`pd.concat`)
* **What it does:** Concatenates separate subsets of data vertically along the row axis to combine data across different time periods.
* **Key Functions & Logic:**
  * **Filtering by Year**: Extracts subsets for specific years (e.g., 2017 and 2018) based on purchase timestamps.
  * **`pd.concat(..., axis=0, keys=[...])`**: Stacks the dataframes vertically and assigns hierarchical index keys (`Year_2017`, `Year_2018`) to track the source of each row.

---

## **Task 6: Integration Decision and Validation**

### **Objective & Concept**

**Integration Plan:** A clear mapping of tables, keys, relationships, and join types before writing code.

**Grain & Repeated Keys:** Understanding why rows repeat (e.g., multiple items or payment installments per order) and aggregating them properly instead of blindly deleting them.

**Avoiding Double-Counting:** Using a step-by-step sequence where child tables are summarized first so revenue and freight totals are not inflated during merges.

### Integration Plan Summary Table

| Parent Table | Child / Related Table | Join Keys (Foreign Key $\rightarrow$ Primary Key) | Expected Relationship | Intended Join Type |
| :--- | :--- | :--- | :--- | :--- |
| `orders` | `customers` | `orders.customer_id` $\rightarrow$ `customers.customer_id` | One-to-One / Many-to-One | Left Join |
| `orders` | `order_items` (Aggregated) | `orders.order_id` $\rightarrow$ `order_items.order_id` | One-to-Many (Summarized to 1-to-1) | Left Join |
| `orders` | `order_payments` (Aggregated) | `orders.order_id` $\rightarrow$ `order_payments.order_id` | One-to-Many (Summarized to 1-to-1) | Left Join |
| `orders` | `order_reviews` (Aggregated) | `orders.order_id` $\rightarrow$ `order_reviews.order_id` | One-to-One (Summarized) | Left Join |

In [ ]:
# 1. Compare row counts before and after integration
initial_order_rows = len(df_orders)
print(f"Row count of orders table before integration: {initial_order_rows:,}")

# Check row counts after building df_master
final_master_rows = len(df_master)
print(f"Row count of master table after integration : {final_master_rows:,}")

# 2. Audit match indicators to check for unmatched records
print("\n--- Unmatched Records Audit (_merge_indicator) ---")
print(df_master['_merge_indicator'].value_counts())

# 3. Investigate repeated keys (Why do duplicates happen in raw child tables?)
print("\n--- Investigating Repeated Keys in order_items ---")
sample_repeated_order = df_items['order_id'].value_counts().head(3)
print("Sample order IDs with multiple items:")
print(sample_repeated_order)

# Verify that pre-aggregation prevented financial double-counting
raw_price_sum = df_items['price'].sum()
merged_price_sum = df_master['order_price_sum'].sum()
print(f"\nRaw total price sum from items table : R$ {raw_price_sum:,.2f}")
print(f"Merged total price sum in master table : R$ {merged_price_sum:,.2f}")
print(f"Do totals match exactly?             : {np.isclose(raw_price_sum, merged_price_sum)}")

## Task 6 explaination

### 1. Row Count Comparison (`len`)
* **What it does:** Measures and compares the total row counts of the core orders table before and after integration to verify structural consistency.
* **Key Functions & Logic:**
  * **`len()`**: Computes the total number of rows in `df_orders` and the final `df_master` dataframe to ensure unexpected row inflation did not occur during the merge process.

### 2. Match Indicator Audit (`value_counts`)
* **What it does:** Audits the merge match results to check for any unmatched records or anomalies across the integrated datasets.
* **Key Functions & Logic:**
  * **`_merge_indicator.value_counts()`**: Summarizes the frequencies of match outcomes (e.g., matching vs. unmatched rows) tracked during the merge operation.

### 3. Repeated Keys Investigation and Financial Integrity Verification (`value_counts`, `sum`, `np.isclose`)
* **What it does:** Explores why duplicate keys occur in transactional child tables (such as multiple items per order) and mathematically verifies that pre-aggregation prevented financial double-counting.
* **Key Functions & Logic:**
  * **`value_counts().head(3)`**: Inspects sample `order_id` frequencies in `df_items` to demonstrate that a single order can contain multiple items.
  * **Financial Summation (`sum()`)**: Compares the raw unaggregated item price sum against the aggregated master price sum.
  * **`np.isclose()`**: Performs a numerical tolerance check to safely confirm that both price totals match exactly, proving that pre-aggregation successfully avoided Cartesian product inflation.

---

### **Detailed Explanations and Answers**

**A. Row Counts, Key Counts, and Unmatched Records**
* **Row Counts:** The base orders table starts with 99,441 rows. After merging with customers, aggregated items, payments, and reviews using left joins, the final df_master table maintains the exact same row count (99,441 rows).

* **Unmatched Records:** Using indicator='_merge_indicator' reveals that a small number of orders (around 613 rows) did not have matching item records. This is expected because some orders in Olist have a status of canceled or unavailable, meaning they were cancelled before items were processed or shipped. It does not represent a data error, but rather reflects real-world operational gaps.

**B. Investigation of Repeated Keys (Duplicates)**
* In raw child tables like order_items and order_payments, order_id appears multiple times. For example, one customer order can contain 3 different products or use 2 payment installments (credit card + voucher).
* **Are these errors?** No. These repeated keys represent valid repeated events (multiple items bought in a single shopping cart or split payments).
* **Handling Decision:** We did not drop these duplicate keys blindly. Instead, we investigated their meaning and aggregated the child tables to 1 row per order_id before merging, which preserves the accurate total spend without creating duplicate table rows.

### **C. Why the Integration Sequence Avoids Double-Counting**
* **The Sequence:** We first joined customer details (1-to-1), then pre-aggregated items and payments to the order level, and finally merged them onto orders.
* **Why it prevents double-counting:** If we join unaggregated item tables and payment tables directly to orders at the same time, SQL/Pandas creates a Cartesian product ($M \times n$), multiplying row counts and artificially inflating total revenue and freight sums. By summarizing child tables first, every order ID remains unique, guaranteeing that total revenue matches the raw item data exactly (R$ 13.59M).

## **Task 7: Measures of Central Tendency**

### **Objective & Concept**

* **Mean (Arithmetic Average):** Calculated by summing all numerical observations and dividing by the total count ($n$). The mean is highly sensitive to extreme values or outliers because every data point directly affects the sum.
* **Median (Middle Observation):** The exact middle value when the dataset is sorted in ascending order. Unlike the mean, the median is robust and resistant to extreme values/outliers.
* **Mode (Most Frequent Value):** The most frequently occurring value or category in a dataset.

**Comparing Mean and Median:** The relationship between the mean and median indicates the presence of skewness or outliers: 
* If $\text{Mean} > \text{Median}$, the distribution is right-skewed (positive skew), pulled upward by high-value outliers (common in e-commerce spend).
* If $\text{Mean} < \text{Median}$, the distribution is left-skewed (negative skew), pulled downward by low-value outliers.

In [ ]:
import numpy as np
import pandas as pd

# Select suitable numerical variables from df_master
variables_to_check = ['order_price_sum', 'delivery_duration_days']

stats_list = []
for var in variables_to_check:
    series = df_master[var].dropna()
    mean_val = series.mean()
    median_val = series.median()
    mode_val = series.mode()
    
    # Take the first mode if multiple exist
    mode_val = mode_val.iloc[0] if not mode_val.empty else np.nan
    
    stats_list.append({
        'Variable': var,
        'Mean': mean_val,
        'Median': median_val,
        'Mode': mode_val
    })

central_tendency_df = pd.DataFrame(stats_list)
print("--- Measures of Central Tendency Summary ---")
display(central_tendency_df.round(2))

## Task 7 explaination

### 1. Iterating Over Numerical Variables (`for` loop, `dropna`)
* **What it does:** Loops through selected numerical columns (`'order_price_sum'`, `'delivery_duration_days'`) in the master dataframe, dropping missing values to ensure clean statistical calculations.
* **Key Functions & Logic:**
  * **`dropna()`**: Removes missing (`NaN`) values from each variable's series so they do not skew the statistical outputs.

### 2. Computing Central Tendency Statistics (`mean`, `median`, `mode`)
* **What it does:** Calculates the mean, median, and mode for each selected numerical column.
* **Key Functions & Logic:**
  * **`.mean()` & `.median()`**: Computes the arithmetic average and the middle value for each series.
  * **`.mode()`**: Finds the most frequently occurring value(s).
  * **Mode Safety Check (`iloc[0] if not mode_val.empty else np.nan`)**: Handles cases where multiple modes exist or where the series is empty by safely selecting the first mode or returning `NaN`.

### 3. Structuring and Displaying Summary Results (`DataFrame`, `round`, `display`)
* **What it does:** Compiles the calculated statistics into a structured summary DataFrame, rounds the figures for presentation clarity, and displays the final table.
* **Key Functions & Logic:**
  * **`pd.DataFrame(stats_list)`**: Converts the list of dictionary statistics into a tabular format.
  * **`.round(2)`**: Rounds all numerical values to two decimal places.
  * **`display()`**: Renders the formatted summary table inside the notebook.

---

## **Task 8: Measures of Dispersion**

### **Objective & Concept**

* **Range:** The distance between the maximum and minimum values in the dataset ($\text{Maximum} - \text{Minimum}$). It gives the total span of the data but is highly sensitive to extreme outliers.

* **Variance ($s^2$):** Measures the average squared distance of data points from the arithmetic mean. Because deviations are squared, variance is expressed in squared units (e.g., $\text{BRL}^2$ for prices or $\text{days}^2$ for delivery durations).

* **Standard Deviation ($s$):** The square root of the variance. It describes the spread around the mean in the original units of the variable (BRL, days), making it directly interpretable.

* **Sample vs. Population Convention:** By default, Python libraries like pandas and NumPy compute sample statistics (using $n-1$ degrees of freedom in the denominator, known as Bessel's correction) to provide an unbiased estimate of the population parameters when working with a subset or sample of data.

In [ ]:
# Select suitable numerical variables from df_master
variables_to_check = ['order_price_sum', 'delivery_duration_days']

dispersion_list = []
for var in variables_to_check:
    series = df_master[var].dropna()
    
    # Calculate range, variance (sample convention with n-1 ddof), and standard deviation
    data_range = series.max() - series.min()
    variance = series.var()
    std_dev = series.std()
    
    dispersion_list.append({
        'Variable': var,
        'Range': data_range,
        'Variance': variance,
        'Std_Dev': std_dev
    })

dispersion_df = pd.DataFrame(dispersion_list)
print("--- Measures of Dispersion Summary ---")
display(dispersion_df.round(2))

## Task 8 explaination

### 1. Iterating Over Numerical Variables and Dropping Nulls (`for` loop, `dropna`)
* **What it does:** Loops through the selected numerical columns (`'order_price_sum'`, `'delivery_duration_days'`) in the master dataframe while ignoring missing values.
* **Key Functions & Logic:**
  * **`dropna()`**: Filters out null values from each column's series to ensure accurate mathematical calculations of spread.

### 2. Computing Dispersion Statistics (`max`, `min`, `var`, `std`)
* **What it does:** Calculates the range, sample variance, and standard deviation for each numerical variable to measure data spread.
* **Key Functions & Logic:**
  * **Range (`series.max() - series.min()`)**: Computes the span between the maximum and minimum values in the dataset.
  * **Variance (`series.var()`)**: Calculates sample variance using Pandas' default sample convention (degrees of freedom $ddof=1$).
  * **Standard Deviation (`series.std()`)**: Computes the square root of the variance to measure typical dispersion from the mean.

### 3. Compiling and Displaying the Dispersion Summary (`DataFrame`, `round`, `display`)
* **What it does:** Stores the computed dispersion metrics in a structured dictionary list, converts it into a summary DataFrame, rounds the numbers, and renders the table.
* **Key Functions & Logic:**
  * **`pd.DataFrame(dispersion_list)`**: Converts the list of dictionaries into a structured tabular format.
  * **`.round(2)`**: Rounds all floating-point dispersion metrics to two decimal places for clear presentation.
  * **`display()`**: Outputs the formatted dispersion summary table inside the notebook.

---

## **Task 9: Percentiles, Quantiles and Five-Number Summary**

### **Objective & Concept**

* **Percentiles & Quantiles:** Percentiles divide a ranked dataset into 100 equal parts, indicating the value below which a given percentage of observations fall (e.g., the 90th percentile means 90% of the data lies below that value). Quantiles (such as quartiles) divide the ordered dataset into equal-sized subgroups.

* **Quartiles & Ordered Observations:** Quartiles divide ordered (ranked) observations into four equal quarters:
* * **First Quartile ($Q_1$):** The 25th percentile (median of the lower half).
* * **Second Quartile ($Q_2$):** The 50th percentile (the median)
* * **Third Quartile ($Q_3$):** The 75th percentile (median of the upper half).
* * **Fourth Quartile ($Q_4$):** The 100th percentile (the maximum value of the dataset, representing the upper boundary).

* **Five-Number Summary:**  A comprehensive set of descriptive statistics summarizing the distribution consisting of: Minimum, $Q_1$, Median, $Q_3$, and Maximum.

* **Interquartile Range (IQR):** Defined as $Q_3 - Q_1$, the IQR measures the spread of the middle 50% of the data. Because it excludes the top and bottom 25% of observations, it is a robust measure of dispersion that is unaffected by extreme outliers.

In [ ]:
# Select suitable numerical variables from df_master
variables_to_check = ['order_price_sum', 'delivery_duration_days']

summary_list = []
for var in variables_to_check:
    series = df_master[var].dropna()
    
    # Calculate percentiles and five-number summary components
    p10 = series.quantile(0.10)
    q1 = series.quantile(0.25)
    median = series.quantile(0.50)
    q3 = series.quantile(0.75)
    p90 = series.quantile(0.90)
    iqr = q3 - q1
    min_val = series.min()
    max_val = series.max()
    
    summary_list.append({
        'Variable': var,
        'Min': min_val,
        '10th_%': p10,
        'Q1 (25%)': q1,
        'Median (50%)': median,
        'Q3 (75%)': q3,
        '90th_%': p90,
        'Max': max_val,
        'IQR': iqr
    })

five_num_df = pd.DataFrame(summary_list)
print("--- Five-Number Summary & Percentiles ---")
display(five_num_df.round(2))

## Task 9 explaination

### 1. Iterating Over Numerical Variables and Dropping Nulls (`for` loop, `dropna`)
* **What it does:** Loops through the chosen numerical columns (`'order_price_sum'`, `'delivery_duration_days'`) in the master dataframe while excluding missing values to ensure accurate distribution metrics.
* **Key Functions & Logic:**
  * **`dropna()`**: Removes any null (`NaN`) entries from each variable's series before computing quantiles.

### 2. Computing Percentiles, Quartiles, and IQR (`quantile`, `min`, `max`)
* **What it does:** Calculates key distributional milestones including minimum, 10th percentile, first quartile (Q1), median, third quartile (Q3), 90th percentile, maximum, and the interquartile range (IQR).
* **Key Functions & Logic:**
  * **`series.quantile(q)`**: Computes the numerical value corresponding to specific fractional thresholds (`0.10`, `0.25`, `0.50`, `0.75`, `0.90`).
  * **Interquartile Range (`q3 - q1`)**: Measures the statistical spread of the middle 50% of the data by subtracting Q1 from Q3.
  * **`min()` & `max()`**: Identifies the absolute boundaries of the data series.

### 3. Compiling and Displaying the Summary Table (`DataFrame`, `round`, `display`)
* **What it does:** Organizes the computed percentiles and summary components into a structured dictionary list, converts it into a DataFrame, rounds the figures, and displays the result.
* **Key Functions & Logic:**
  * **`pd.DataFrame(summary_list)`**: Converts the compiled metric list into a clean tabular structure.
  * **`.round(2)`**: Rounds all floating-point values to two decimal places for readable reporting.
  * **`display()`**: Renders the formatted five-number summary and percentile table inside the notebook.


---

## **Task 10: Skewness and Kurtosis**

### **Objective & Concept**

* **Skewness (Asymmetry):** Measures the asymmetry of a distribution around its mean.
* * **Positive Skew (Right-Skewed):** The right tail is longer or fatter; extreme values pull the mean above the median ($\text{Mean} > \text{Median}$). This is common in e-commerce monetary metrics like order value or freight.
* * **Negative Skew (Left-Skewed):** The left tail is longer; extreme low values pull the mean below the median ($\text{Mean} < \text{Median}$).
* * **Approximately Symmetric:** The distribution is balanced on both sides, with the mean and median roughly equal ($\text{Mean} \approx \text{Median}$).

* **Kurtosis (Tail Behavior & Peakedness):** Measures the relative "tailedness" of a distribution compared to a normal distribution.
* * **Leptokurtic (High Kurtosis):** Fat tails and a sharp peak, indicating a higher propensity for extreme outliers.
* * **Platykurtic (Low Kurtosis):** Thin tails and a flatter peak.
* * **Software Convention (Excess vs. Pearson Kurtosis):** Standard libraries like Pandas (.kurt()) and SciPy report excess kurtosis by default. In excess kurtosis, a standard normal distribution has a reference value of $0$ (whereas Pearson kurtosis sets the normal reference at $3$).

* **Practical Implications:**
* * **Choice of Summary Measure:** When data is heavily skewed or leptokurtic, the mean and standard deviation become distorted by outliers. In such cases, robust statistics like the median and IQR should be prioritized.
* * **Distribution Assumptions:** Parametric statistical tests and models often assume normality. High skewness or kurtosis violates these assumptions, necessitating data transformations (e.g., log transformations).
* * **Outlier Investigation:** High kurtosis highlights the presence of extreme values that require investigation rather than blind removal, as they often represent genuine operational realities (e.g., high-value wholesale or bulk orders).

In [ ]:
# Select suitable numerical variables from df_master
variables_to_check = ['order_price_sum', 'delivery_duration_days']

skew_kurt_list = []
for var in variables_to_check:
    series = df_master[var].dropna()
    
    skewness = series.skew()
    # pandas .kurt() calculates excess kurtosis by default (normal distribution = 0)
    excess_kurtosis = series.kurt()
    
    skew_kurt_list.append({
        'Variable': var,
        'Skewness': skewness,
        'Excess_Kurtosis': excess_kurtosis
    })

skew_kurt_df = pd.DataFrame(skew_kurt_list)
print("--- Skewness and Kurtosis Summary ---")
display(skew_kurt_df.round(2))

## Task 10 explaination

### 1. Iterating Over Numerical Variables and Dropping Nulls (`for` loop, `dropna`)
* **What it does:** Loops through the selected numerical columns (`'order_price_sum'`, `'delivery_duration_days'`) in the master dataframe while ignoring missing values.
* **Key Functions & Logic:**
  * **`dropna()`**: Removes null entries from each column's series to ensure accurate distribution shape calculations.

### 2. Computing Distribution Shape Statistics (`skew`, `kurt`)
* **What it does:** Calculates the skewness (asymmetry) and excess kurtosis (tailedness) for each numerical variable.
* **Key Functions & Logic:**
  * **`series.skew()`**: Measures the asymmetry of the data distribution around the mean (positive or negative tilt).
  * **`series.kurt()`**: Computes excess kurtosis using Pandas' default convention, where a normal distribution has a baseline excess kurtosis of 0 (measuring the presence of heavy tails and outliers).

### 3. Compiling and Displaying the Shape Summary (`DataFrame`, `round`, `display`)
* **What it does:** Organizes the computed skewness and kurtosis metrics into a structured dictionary list, converts it into a DataFrame, rounds the figures, and displays the result.
* **Key Functions & Logic:**
  * **`pd.DataFrame(skew_kurt_list)`**: Converts the compiled metric list into a clean tabular structure.
  * **`.round(2)`**: Rounds all floating-point values to two decimal places for readable reporting.
  * **`display()`**: Renders the formatted skewness and kurtosis summary table inside the notebook.

---

## **Task 11: Common Distributions and Visualizations**

### Recommended Python Libraries for Distribution Visualization
* **What it does:** Specifies the standard data visualization libraries used in Python to build histograms, density curves, and box plots for distribution analysis.
* **Key Libraries & Functions:**
  * **`matplotlib.pyplot` (`plt`)**: The foundational plotting library used for creating custom figures, axes, and basic histograms (`plt.hist`).
  * **`seaborn` (`sns`)**: A high-level interface built on top of Matplotlib that simplifies statistical graphics plotting, such as kernel density estimation plots (`sns.kdeplot`), distribution plots (`sns.histplot`), and box plots (`sns.boxplot`).

## **Task 12: Empirical Rule (68–95–99.7 Rule)**

### **Objective & Concept**

* **The Empirical Rule (Three-Sigma Rule)**: For any dataset that follows an **approximately normal, bell-shaped, single-peaked distribution**, the spread of data relative to the mean and standard deviation $(\sigma)$ follows a strict predictable proportion:
  * **$\text{Mean} \pm 1\sigma$**: Approximately **68%** of all observations fall within one standard deviation.
  * **$\text{Mean} \pm 2\sigma$**: Approximately **95%** of all observations fall within two standard deviations.
  * **$\text{Mean} \pm 3\sigma$**: Approximately **99.7%** of all observations fall within three standard deviations.

* **Why It Fails for Non-Normal Data**: The empirical rule **does not work well** for highly skewed, heavy-tailed, or bimodal distributions because the core underlying assumptions—symmetry, single-peakedness, and bell-shaped normality—are violated. For instance, in heavily right-skewed e-commerce financial data, the mean is artificially dragged upward by extreme high-value outliers, meaning the $\text{Mean} - 1\sigma$ lower bound might drop below zero, and a much higher percentage (often 85%+) will bunch up below the mean.

* **Empirical vs. Simulation Approach**: Because real-world monetary and operational variables (like Olist order prices) are often skewed rather than strictly normal, we demonstrate the rule using a **clean simulated normal distribution** alongside a **skewed real-world comparison** to highlight why the rule fails when normality assumptions are broken.

In [ ]:
# 1. Simulated Normal Distribution (where the Empirical Rule holds true)
np.random.seed(42)
normal_data = np.random.normal(loc=100.0, scale=15.0, size=10000)

mean_n = np.mean(normal_data)
std_n = np.std(normal_data, ddof=1)

pct_1sd_n = np.mean((normal_data >= mean_n - 1 * std_n) & (normal_data <= mean_n + 1 * std_n)) * 100
pct_2sd_n = np.mean((normal_data >= mean_n - 2 * std_n) & (normal_data <= mean_n + 2 * std_n)) * 100
pct_3sd_n = np.mean((normal_data >= mean_n - 3 * std_n) & (normal_data <= mean_n + 3 * std_n)) * 100

print("--- 1. Simulated Normal Distribution Empirical Rule Check ---")
print(f"Mean: {mean_n:.2f} | Std Dev: {std_n:.2f}")
print(f"Within ±1 SD : {pct_1sd_n:.2f}% (Expected: ~68%)")
print(f"Within ±2 SD : {pct_2sd_n:.2f}% (Expected: ~95%)")
print(f"Within ±3 SD : {pct_3sd_n:.2f}% (Expected: ~99.7%)")

# 2. Skewed Distribution (Real Olist Order Price Sum - where Empirical Rule breaks down)
skewed_data = df_master['order_price_sum'].dropna()
mean_s = skewed_data.mean()
std_s = skewed_data.std()

pct_1sd_s = np.mean((skewed_data >= mean_s - 1 * std_s) & (skewed_data <= mean_s + 1 * std_s)) * 100
pct_2sd_s = np.mean((skewed_data >= mean_s - 2 * std_s) & (skewed_data <= mean_s + 2 * std_s)) * 100
pct_3sd_s = np.mean((skewed_data >= mean_s - 3 * std_s) & (skewed_data <= mean_s + 3 * std_s)) * 100

print("\n--- 2. Skewed Distribution (Order Price Sum) Empirical Rule Check ---")
print(f"Mean: {mean_s:.2f} | Std Dev: {std_s:.2f}")
print(f"Within ±1 SD : {pct_1sd_s:.2f}% (Expected: ~68%)")
print(f"Within ±2 SD : {pct_2sd_s:.2f}% (Expected: ~95%)")
print(f"Within ±3 SD : {pct_3sd_s:.2f}% (Expected: ~99.7%)")

## Task 12 explaination

### 1. Simulating a Normal Distribution and Checking the Empirical Rule (`np.random.normal`, `mean`, `std`)
* **What it does:** Generates a synthetic normally distributed dataset to test and confirm the standard Empirical Rule (68-95-99.7% rule).
* **Key Functions & Logic:**
  * **`np.random.seed(42)`**: Sets a pseudo-random seed for reproducibility.
  * **`np.random.normal(loc=100.0, scale=15.0, size=10000)`**: Generates 10,000 data points with a mean of 100 and a standard deviation of 15.
  * **Empirical Rule Percentage Calculations (`np.mean(...) * 100`)**: Evaluates the percentage of data points falling within $\pm 1$, $\pm 2$, and $\pm 3$ standard deviations of the mean, comparing them against the theoretical expectations (~68%, ~95%, ~99.7%).

### 2. Evaluating the Empirical Rule on Real Skewed Data (`dropna`, Boolean Masking)
* **What it does:** Tests the Empirical Rule on a real, right-skewed dataset (`order_price_sum` from `df_master`) to demonstrate how the rule breaks down when distributions are non-normal.
* **Key Functions & Logic:**
  * **`dropna()`**: Cleans the target series by removing missing values.
  * **Boolean Masking (`(skewed_data >= ...) & (skewed_data <= ...)`)**: Calculates the actual proportion of skewed data falling within 1, 2, and 3 standard deviations of the mean.
  * **Comparative Analysis**: Prints the empirical percentages against the normal distribution benchmarks to highlight the impact of data skewness on standard deviation intervals.

---

## **Task 13: Sampling Concepts and the Central Limit Theorem (CLT)**

### **Objective & Concept**

* **Population vs. Sample**: The entire collection of all observations is the *population* (e.g., all 99,441 Olist orders), while a *sample* is a subset drawn from that population.

* **The Central Limit Theorem (CLT)**: States that when independent random samples of size $n$ (typically $n \ge 30$) are drawn from *any* population (regardless of whether the underlying population is skewed, bimodal, or uniform), the sampling distribution of the sample means will approximate a **normal distribution**. Furthermore, the mean of the sample means equals the population mean, and the standard error of the mean is given by $\sigma_{\bar{x}} = \frac{\sigma}{\sqrt{n}}$.

* **Practical EDA Application**: Allows analysts to apply parametric statistical tests and confidence intervals on sample data even when the underlying e-commerce data (like customer spend or order value) is heavily skewed.

In [ ]:
# Using a heavily skewed population variable: order_price_sum
population_data = df_master['order_price_sum'].dropna()

# Draw multiple samples of size n=50 and compute sample means
np.random.seed(42)
sample_sizes = 50
num_iterations = 1000
sample_means = [np.mean(np.random.choice(population_data, size=sample_sizes, replace=True)) for _ in range(num_iterations)]

print(f"Population Mean: {population_data.mean():.2f}")
print(f"Mean of 1,000 Sample Means (n={sample_sizes}): {np.mean(sample_means):.2f}")
print(f"Standard Error of the Mean: {np.std(sample_means, ddof=1):.2f}")

## Task 13 explaination

### 1. Preparing the Skewed Population Data (`dropna`)
* **What it does:** Extracts a heavily skewed population variable (`order_price_sum`) from the master dataframe and removes missing values to form the baseline population.
* **Key Functions & Logic:**
  * **`dropna()`**: Cleans the target series to ensure sampling operations do not encounter `NaN` values.

### 2. Simulating Repeated Sampling (`np.random.seed`, `np.random.choice`, List Comprehension)
* **What it does:** Draws 1,000 random samples of size $n=50$ with replacement from the skewed population and calculates the mean for each sample.
* **Key Functions & Logic:**
  * **`np.random.seed(42)`**: Sets a pseudo-random seed to ensure reproducibility of the simulation results.
  * **`np.random.choice(..., size=50, replace=True)`**: Randomly selects 50 data points from the population dataset with replacement.
  * **List Comprehension & `np.mean()`**: Iterates through 1,000 iterations to compute and store the sample mean for each drawn sample.

### 3. Computing Summary Statistics and Evaluating CLT (`mean`, `std`)
* **What it does:** Calculates the true population mean, the average of all 1,000 sample means, and the standard error of the mean to demonstrate the Central Limit Theorem in action.
* **Key Functions & Logic:**
  * **Population vs. Sample Mean Comparison**: Computes `population_data.mean()` and `np.mean(sample_means)` to show that the mean of the sample means closely approximates the true population mean even when the underlying population is heavily skewed.
  * **Standard Error (`np.std(..., ddof=1)`)**: Computes the standard deviation of the sample means (with degrees of freedom $ddof=1$) to measure the standard error of the mean.

---

## **Task 14:  Covariance and Correlation**

### **Objective & Concept**

* **Covariance**: Measures the directional relationship between two numerical variables. A positive covariance indicates variables move together; negative covariance means they move in opposite directions. However, covariance depends heavily on data units and scales, making direct comparisons difficult.

* **Pearson Correlation Coefficient ($r$)**: Measures the **linear** strength and direction between two continuous variables, ranging from $-1$ to $+1$. Assumes approximate normality, linearity, and homoscedasticity.

* **Spearman Rank Correlation ($\rho$)**: A non-parametric measure that evaluates **monotonic** relationships by ranking the data values rather than using raw numbers. Highly robust against extreme outliers and skewed distributions, making it ideal for e-commerce monetary data.

* **Correlation Matrix & Correlation vs. Causation**: 
  * **Correlation Matrix**: A tabular layout displaying correlation coefficients between multiple pairs of variables.
  * **Correlation vs. Causation**: *Correlation does not imply causation*. Two variables moving together may be driven by a third confounding variable rather than a direct cause-and-effect link.

In [ ]:
# Select suitable numerical variables for correlation and covariance check
corr_vars = ['order_price_sum', 'order_freight_sum', 'delivery_duration_days', 'total_payment_value']
sub_df = df_master[corr_vars].dropna()

# 1. Covariance Matrix (Directional relationship, unit-dependent)
covariance_matrix = sub_df.cov()
print("--- Covariance Matrix ---")
display(covariance_matrix.round(2))

# 2. Pearson Correlation Matrix (Linear strength and direction)
pearson_matrix = sub_df.corr(method='pearson')
print("\n--- Pearson Correlation Matrix ---")
display(pearson_matrix.round(2))

# 3. Spearman Correlation Matrix (Monotonic relationship - robust to skewness/outliers)
spearman_matrix = sub_df.corr(method='spearman')
print("\n--- Spearman Correlation Matrix ---")
display(spearman_matrix.round(2))

## Task 14 explaination

### 1. Selecting Variables and Cleaning Data (`dropna`)
* **What it does:** Selects key numerical variables (`'order_price_sum'`, `'order_freight_sum'`, `'delivery_duration_days'`, `'total_payment_value'`) from the master dataframe and removes missing values.
* **Key Functions & Logic:**
  * **`dropna()`**: Filters out null entries across the subset DataFrame to ensure valid mathematical calculations for covariance and correlation.

### 2. Computing the Covariance Matrix (`cov`)
* **What it does:** Calculates the covariance matrix to measure the directional joint variability between pairs of numerical variables.
* **Key Functions & Logic:**
  * **`.cov()`**: Computes pairwise sample covariances; note that the output scale is unit-dependent and sensitive to the magnitude of the variables.

### 3. Computing Pearson and Spearman Correlation Matrices (`corr`)
* **What it does:** Measures the strength and direction of associations between variables using both linear (Pearson) and monotonic rank-based (Spearman) methods.
* **Key Functions & Logic:**
  * **`method='pearson'`**: Evaluates linear relationships between variables.
  * **`method='spearman'`**: Evaluates monotonic relationships by ranking values, making it robust against heavy skewness and outliers.

### 4. Compiling and Displaying Matrices (`round`, `display`)
* **What it does:** Rounds all computed covariance and correlation coefficients to two decimal places and renders the matrices inside the notebook.
* **Key Functions & Logic:**
  * **`.round(2)`**: Formats the matrix floating-point numbers for clear readability.
  * **`display()`**: Outputs the formatted covariance, Pearson, and Spearman matrices.

---

## **Task 15: Outlier Detection**

### **Objective & Concept**

* **What is an Outlier?**: An observation that lies an abnormal distance from other values in a random sample from a population. In e-commerce, outliers often represent legitimate high-value wholesale transactions rather than data entry errors.

* **Detection Methods**:
  * **Z-Score Method**: Identifies points where $\vert{}z\vert{} > 3$ (how many standard deviations a point is from the mean). Best suited for normally distributed data.
  * **IQR (Interquartile Range) Method**: Highly robust and preferred for skewed e-commerce distributions. It identifies points outside the fences defined by:
$$\text{Lower Bound} = Q_1 - 1.5 \times \text{IQR}$$
$$\text{Upper Bound} = Q_3 + 1.5 \times \text{IQR}$$


* **Handling / Treatment Strategies**:
  1. **Retain**: Keep genuine extreme values if they reflect real business activity.
  2. **Transformation**: Apply log or square root scaling to compress extreme right tails.
  3. **Capping / Winsorizing**: Cap extreme values at specific percentile thresholds (e.g., 99th percentile) instead of deleting them.
  4. **Removal**: Drop rows strictly if verified as corrupted records or data errors.

In [ ]:
# Target variable for outlier analysis
series = df_master['order_price_sum'].dropna()

# 1. IQR Method
q1 = series.quantile(0.25)
q3 = series.quantile(0.75)
iqr = q3 - q1

lower_fence_iqr = q1 - 1.5 * iqr
upper_fence_iqr = q3 + 1.5 * iqr

outliers_iqr = series[(series < lower_fence_iqr) | (series > upper_fence_iqr)]
print(f"--- IQR Outlier Detection ---")
print(f"Lower Fence: {lower_fence_iqr:.2f} | Upper Fence: {upper_fence_iqr:.2f}")
print(f"Outliers detected (IQR): {len(outliers_iqr):,} ({(len(outliers_iqr)/len(series))*100:.2f}%)")

# 2. Z-Score Method
mean_val = series.mean()
std_val = series.std()
z_scores = (series - mean_val) / std_val
outliers_z = series[np.abs(z_scores) > 3]
print(f"\n--- Z-Score Outlier Detection (|z| > 3) ---")
print(f"Outliers detected (Z-score): {len(outliers_z):,} ({(len(outliers_z)/len(series))*100:.2f}%)")

# 3. Treatment Strategy: Capping / Winsorizing at the 99th Percentile
cap_threshold = series.quantile(0.99)
series_winsorized = series.clip(upper=cap_threshold)
print(f"\n--- Winsorization ---")
print(f"Capped upper limit at 99th percentile ({cap_threshold:.2f}): max value is now {series_winsorized.max():.2f}")

## Task 15 explaination

### 1. Preparing Data and IQR Outlier Detection (`dropna`, `quantile`, Boolean Masking)
* **What it does:** Isolates the target numerical variable (`order_price_sum`), computes statistical quartiles and the interquartile range, establishes lower/upper fences, and identifies non-parametric outliers.
* **Key Functions & Logic:**
  * **`dropna()`**: Removes null values from the target series.
  * **`quantile()`**: Computes the first quartile ($Q1$ at 25%) and third quartile ($Q3$ at 75%) to calculate $\text{IQR} = Q3 - Q1$.
  * **Fences (`1.5 * iqr`)**: Establishes robust outlier boundaries using standard Tukey fences ($Q1 - 1.5 \times \text{IQR}$ and $Q3 + 1.5 \times \text{IQR}$).
  * **Boolean Filtering**: Filters out-of-bounds values to calculate total outlier count and percentage share.

### 2. Z-Score Outlier Detection (`mean`, `std`, Absolute Z-Scores)
* **What it does:** Identifies parametric outliers by measuring how many standard deviations each data point lies away from the mean, flagging points where $|z| > 3$.
* **Key Functions & Logic:**
  * **`mean()` & `std()`**: Calculates the central tendency and dispersion needed for standardization.
  * **Z-Score Formula (`(series - mean_val) / std_val`)**: Converts raw values into standard deviation units.
  * **`np.abs(z_scores) > 3`**: Filters points exceeding a threshold of 3 standard deviations in either direction as outliers.

### 3. Outlier Treatment via Winsorization (`quantile`, `clip`)
* **What it does:** Caps extreme upper-bound values at a specified percentile threshold to mitigate the skewing impact of outliers without dropping rows.
* **Key Functions & Logic:**
  * **`quantile(0.99)`**: Determines the ceiling threshold at the 99th percentile.
  * **`series.clip(upper=cap_threshold)`**: Truncates any values exceeding the 99th percentile down to the cap threshold limit.

---

## **Conclusion & Executive Summary**

### **1. Key Analytical Findings & Insights**

* **Data Integrity & Integration**: Through rigorous grain management, key uniqueness checks, and pre-aggregation of child tables (`order_items`, `order_payments`), we successfully built a unified `df_master` table (99,441 orders) without Cartesian product inflation or revenue distortion.

* **Distribution & Skewness**: Monetary variables (`order_price_sum`, `total_payment_value`) and operational metrics (`delivery_duration_days`) exhibit pronounced **positive skewness and leptokurtic behavior (heavy tails)**. This confirms that the e-commerce platform handles a high volume of low-to-moderate transactions alongside a long right tail of high-value wholesale or bulk orders.

* **Central Tendency & Spread**: Due to extreme right-tail skewness, the **mean** is pulled significantly higher than the **median**. Consequently, robust summary statistics—such as the median, Interquartile Range (IQR), and Spearman rank correlation—proved far more reliable than parametric counterparts for characterizing typical customer behavior.

* **Correlations & Relationships**: Correlation analyses revealed strong monotonic relationships between order value and payment sums, while highlighting that logistics delays (delivery duration) operate independently of product pricing, pointing to regional fulfillment bottlenecks rather than product-specific friction.

* **Outlier Dynamics**: Outliers identified via the IQR method ($Q_3 + 1.5 \times \text{IQR}$) were verified not as data entry errors, but as genuine high-value commercial transactions. Treatment strategies like Winsorization and log-transformations successfully mitigated skewness while preserving underlying business value.


---

### **2. Business & Operational Recommendations**

* **Logistics Optimization**: Given the spread in delivery durations and regional distribution disparities (e.g., fulfillment delays in peripheral states compared to core hubs like São Paulo), Olist should establish regional fulfillment micro-hubs to reduce transit times.

* **Payment Processing Strategy**: With credit cards and boleto dominating transaction types, financial infrastructure must ensure seamless high-concurrency processing for installments and voucher integrations.

* **Risk & Fraud Monitoring**: The heavy-tailed distribution of order values necessitates automated anomaly detection on upper-percentile transactions to safeguard against fraud without penalizing legitimate high-tier wholesale buyers.

---

### **3. Future Work**

* **Predictive Modeling**: Leverage the engineered `df_master` dataset to build machine learning models predicting delivery delays and customer churn.

* **Customer Segmentation**: Implement K-Means clustering or RFM (Recency, Frequency, Monetary) analysis on customer states and purchasing behavior to drive targeted marketing campaigns.